# ExamAce: investigating QAOA for personalized study scheduling

**Team:** Alex Zhao, Risi Chen, Charlie Chen and Hank Ruan.

Our purpose is to investigate whether QAOA can select useful study sessions under a student's scheduling constraints, and to understand which design choices affect its performance. ExamAce uses a student's diagnostic scores and availability to define a concrete optimization problem, then compares quantum simulation with conventional search methods.

This notebook accompanies the application and presentation. The default run recomputes summaries from embedded, previously recorded benchmark data. It does **not** silently present saved quantum results as fresh computation. Section 6 can execute the repository's real Qiskit implementation, and an optional section can regenerate all 27 runs.

**Quick start:** Open this file in VS Code with the Python and Jupyter extensions, select a Python kernel, and choose **Run All**. The default analysis needs only Python's standard library. For a fresh quantum run, place this notebook in the cloned repository's root, install the project dependencies and `ipykernel`, set `RUN_LIVE = True` below, and run all cells again. Save the notebook after execution so its outputs remain visible to judges.


In [1]:
import json
import sys
import subprocess
from pathlib import Path
from statistics import mean, median

RUN_LIVE = True  # True executes one fresh 12-qubit Qiskit comparison.
RUN_FULL_BENCHMARK = False  # True also regenerates all 27 runs; requires RUN_LIVE.
SOURCE_COMMIT = "ae69d31018730050a25592d46b6d0fbebd18f03f"
print("Mode:", "fresh Qiskit run + saved-data analysis" if RUN_LIVE else "saved-data analysis only")
print("Python:", sys.version.split()[0])


Mode: fresh Qiskit run + saved-data analysis
Python: 3.12.16


## 1. How the student's needs become a scheduling problem

The fictional course contains derivatives, integrals and series. A six-question diagnostic provides an initial proficiency estimate for each topic. Topic weights and exam urgency then influence the value assigned to review and practice sessions. This is a designed planning score, not a measured gain in grades or learning.

Each candidate session has a fixed topic, activity, start time and duration. Selecting or skipping each session forms a binary decision, so 18 candidates have 262,144 possible subsets. A valid schedule must fit availability windows and deadlines, leave space between sessions, satisfy review requirements when enabled, and include breaks within the time budget. The model reduces the value of repeated activities to discourage excessive repetition.

The current simulator first calculates the score and violations of every subset. This exhaustive preprocessing, as well as statevector simulation, limits scalability. The number of combinations alone does not establish a quantum advantage, since stronger classical solvers need not enumerate every subset.


In [2]:
# Embedded copies allow judges to inspect the evidence without a private-repo login.
# These are recorded data, not newly generated quantum outputs.
DATA = json.loads('{"benchmark":{"model_version":3,"cases":[{"case":"weak_integrals_12","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_integrals_15","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_integrals_18","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_12","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_15","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_18","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_12","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_15","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_18","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}}],"settings":{"initial_state":"budget-aware independent ry rotations","seeds":[7,11,23],"p":1,"max_evaluations":30,"shots":1024},"results":[{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":7,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":3.8612381442257346,"uniform":3.2503967202775144,"feasible_rate":0.1318359375,"uniform_feasible_rate":0.0234375,"gap":0.18780011111715567,"exact_ms":0.6658330094069242,"qaoa_ms":39.57899985834956,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":11,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":4.04903825534289,"uniform":3.593685474673065,"feasible_rate":0.140625,"uniform_feasible_rate":0.0244140625,"gap":0.0,"exact_ms":0.48166606575250626,"qaoa_ms":39.38774997368455,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":23,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":4.04903825534289,"uniform":3.680806462943449,"feasible_rate":0.1181640625,"uniform_feasible_rate":0.03125,"gap":0.0,"exact_ms":0.3256250638514757,"qaoa_ms":40.74058309197426,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":7,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":3.689133493438129,"uniform":5.4090034511404355,"feasible_rate":0.0537109375,"uniform_feasible_rate":0.025390625,"gap":2.02238431080815,"exact_ms":3.8086248096078634,"qaoa_ms":217.08941692486405,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":11,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":5.107748064052698,"uniform":4.0749993791089665,"feasible_rate":0.0673828125,"uniform_feasible_rate":0.0205078125,"gap":0.6037697401935809,"exact_ms":3.1626669224351645,"qaoa_ms":211.5951671730727,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":23,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":4.937708280436755,"uniform":4.193889295889295,"feasible_rate":0.0810546875,"uniform_feasible_rate":0.013671875,"gap":0.7738095238095237,"exact_ms":3.102292073890567,"qaoa_ms":210.26070811785758,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":7,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":6.1595178042462795,"uniform":6.864454694164283,"feasible_rate":0.041015625,"uniform_feasible_rate":0.0107421875,"gap":1.9792424908424948,"exact_ms":31.883457908406854,"qaoa_ms":2446.1079160682857,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":11,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":6.203684167684166,"uniform":6.896613460613459,"feasible_rate":0.0634765625,"uniform_feasible_rate":0.0078125,"gap":1.9350761274046082,"exact_ms":36.09108296222985,"qaoa_ms":2412.754832999781,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":23,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":6.507233127137762,"uniform":5.633484965567156,"feasible_rate":0.0380859375,"uniform_feasible_rate":0.0126953125,"gap":1.6315271679510124,"exact_ms":36.254792008548975,"qaoa_ms":2475.499999942258,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":7,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":5.070688514782901,"uniform":4.859542917762095,"feasible_rate":0.126953125,"uniform_feasible_rate":0.0234375,"gap":0.0035002138385964443,"exact_ms":0.6880420260131359,"qaoa_ms":41.77450016140938,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":11,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":5.070688514782901,"uniform":5.070688514782901,"feasible_rate":0.1591796875,"uniform_feasible_rate":0.0244140625,"gap":0.0035002138385964443,"exact_ms":0.3298749215900898,"qaoa_ms":44.83358399011195,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":23,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":5.074188728621498,"uniform":5.00592061819459,"feasible_rate":0.1162109375,"uniform_feasible_rate":0.03125,"gap":0.0,"exact_ms":0.43633393943309784,"qaoa_ms":41.282375110313296,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":7,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":6.858903866179374,"uniform":7.397927130201102,"feasible_rate":0.0830078125,"uniform_feasible_rate":0.025390625,"gap":0.5986089466089473,"exact_ms":3.002375131472945,"qaoa_ms":254.2455419898033,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":11,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":6.989414309688281,"uniform":6.942422346855117,"feasible_rate":0.05859375,"uniform_feasible_rate":0.0205078125,"gap":0.4680985031000402,"exact_ms":3.146165981888771,"qaoa_ms":244.1591250244528,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":23,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":6.396735856735855,"uniform":6.720006512006511,"feasible_rate":0.080078125,"uniform_feasible_rate":0.013671875,"gap":1.0607769560524662,"exact_ms":4.275167128071189,"qaoa_ms":244.37050009146333,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":7,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":10.300167326331707,"uniform":12.263124772943948,"feasible_rate":0.05078125,"uniform_feasible_rate":0.0107421875,"gap":2.514740922395717,"exact_ms":50.5322499666363,"qaoa_ms":3167.955958051607,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":11,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":10.102262922262922,"uniform":12.222445570664746,"feasible_rate":0.0556640625,"uniform_feasible_rate":0.0078125,"gap":2.712645326464502,"exact_ms":37.93933289125562,"qaoa_ms":2869.2930000834167,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":23,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":10.142942124542124,"uniform":10.506143654417626,"feasible_rate":0.0419921875,"uniform_feasible_rate":0.0126953125,"gap":2.6719661241852997,"exact_ms":36.40229091979563,"qaoa_ms":2666.8391248676926,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":7,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":4.990701005641402,"uniform":4.919081958022355,"feasible_rate":0.12890625,"uniform_feasible_rate":0.0234375,"gap":1.2257320427916456,"exact_ms":0.8275001309812069,"qaoa_ms":46.05620796792209,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":11,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":4.990701005641402,"uniform":6.216433048433047,"feasible_rate":0.0673828125,"uniform_feasible_rate":0.0244140625,"gap":1.2257320427916456,"exact_ms":0.35795802250504494,"qaoa_ms":58.934584027156234,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":23,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":5.322279202279201,"uniform":4.885948717948717,"feasible_rate":0.1171875,"uniform_feasible_rate":0.03125,"gap":0.8941538461538459,"exact_ms":0.3367920871824026,"qaoa_ms":43.38895808905363,"evaluations":30,"initial_selection_probability":0.3333333333333333,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":7,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":5.515578347578347,"uniform":5.570780626780627,"feasible_rate":0.0908203125,"uniform_feasible_rate":0.025390625,"gap":1.8787008547008535,"exact_ms":3.8458749186247587,"qaoa_ms":250.06520794704556,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":11,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":6.216433048433047,"uniform":6.216433048433047,"feasible_rate":0.0908203125,"uniform_feasible_rate":0.0205078125,"gap":1.1778461538461533,"exact_ms":4.423875128850341,"qaoa_ms":264.0237498562783,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":23,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":6.0129458689458675,"uniform":6.7036125356125345,"feasible_rate":0.0810546875,"uniform_feasible_rate":0.013671875,"gap":1.381333333333333,"exact_ms":4.725833889096975,"qaoa_ms":240.758917061612,"evaluations":29,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":7,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":8.703006267806266,"uniform":6.840328205128205,"feasible_rate":0.041015625,"uniform_feasible_rate":0.0107421875,"gap":0.0,"exact_ms":39.96525006368756,"qaoa_ms":3370.0697920285165,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":11,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":6.216433048433047,"uniform":6.819988603988603,"feasible_rate":0.044921875,"uniform_feasible_rate":0.0078125,"gap":2.486573219373219,"exact_ms":45.708667021244764,"qaoa_ms":3265.863207867369,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":23,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":5.4589948717948715,"uniform":7.926472934472935,"feasible_rate":0.056640625,"uniform_feasible_rate":0.0126953125,"gap":3.2440113960113948,"exact_ms":45.13966618105769,"qaoa_ms":3468.7132919207215,"evaluations":30,"initial_selection_probability":0.35,"code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false}]},"initial_benchmark":{"model_version":3,"cases":[{"case":"weak_integrals_12","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_integrals_15","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_integrals_18","answers":{"Q-T1-D1":"A","Q-T1-D2":"B","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":1.0,"T2":0.0,"T3":0.5},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_12","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_15","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"mixed_gaps_18","answers":{"Q-T1-D1":"A","Q-T1-D2":"A","Q-T2-D3":"A","Q-T2-D4":"B","Q-T3-D5":"A","Q-T3-D6":"A"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_12","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":240,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_15","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":360,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}},{"case":"weak_derivatives_18","answers":{"Q-T1-D1":"B","Q-T1-D2":"A","Q-T2-D3":"B","Q-T2-D4":"B","Q-T3-D5":"C","Q-T3-D6":"D"},"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":0.0,"T2":0.5,"T3":1.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}}}],"settings":{"seeds":[7,11,23],"p":1,"max_evaluations":30,"shots":1024},"results":[{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":7,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":4.04903825534289,"uniform":3.2503967202775144,"feasible_rate":0.0302734375,"uniform_feasible_rate":0.0234375,"gap":0.0,"exact_ms":0.7096249610185623,"qaoa_ms":38.373625138774514,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":11,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":3.2503967202775144,"uniform":3.593685474673065,"feasible_rate":0.04296875,"uniform_feasible_rate":0.0244140625,"gap":0.7986415350653759,"exact_ms":0.34162518568336964,"qaoa_ms":36.93204186856747,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_12","choices":12,"budget_minutes":240,"seed":23,"exact":4.04903825534289,"greedy":4.04903825534289,"qaoa":3.593685474673065,"uniform":3.680806462943449,"feasible_rate":0.0751953125,"uniform_feasible_rate":0.03125,"gap":0.4553527806698252,"exact_ms":0.3364169970154762,"qaoa_ms":38.046207977458835,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":7,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":5.4090034511404355,"uniform":5.4090034511404355,"feasible_rate":0.0244140625,"uniform_feasible_rate":0.025390625,"gap":0.3025143531058436,"exact_ms":3.5620410926640034,"qaoa_ms":202.3362920153886,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":11,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":4.027670117807103,"uniform":4.0749993791089665,"feasible_rate":0.0107421875,"uniform_feasible_rate":0.0205078125,"gap":1.6838476864391758,"exact_ms":2.623375039547682,"qaoa_ms":192.87629215978086,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_15","choices":15,"budget_minutes":360,"seed":23,"exact":5.711517804246279,"greedy":5.711517804246279,"qaoa":5.328868533658606,"uniform":4.193889295889295,"feasible_rate":0.021484375,"uniform_feasible_rate":0.013671875,"gap":0.38264927058767295,"exact_ms":2.964666113257408,"qaoa_ms":199.0634580142796,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":7,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":7.308750834350833,"uniform":6.864454694164283,"feasible_rate":0.0068359375,"uniform_feasible_rate":0.0107421875,"gap":0.8300094607379416,"exact_ms":29.410041868686676,"qaoa_ms":2188.760499935597,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":11,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":6.2006451703547585,"uniform":6.896613460613459,"feasible_rate":0.0087890625,"uniform_feasible_rate":0.0078125,"gap":1.9381151247340158,"exact_ms":28.53775001130998,"qaoa_ms":2338.733916869387,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_integrals_18","choices":18,"budget_minutes":600,"seed":23,"exact":8.138760295088774,"greedy":8.138760295088774,"qaoa":6.29826059399758,"uniform":5.633484965567156,"feasible_rate":0.0087890625,"uniform_feasible_rate":0.0126953125,"gap":1.8404997010911943,"exact_ms":34.9780002143234,"qaoa_ms":2243.5396248474717,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":7,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":5.046864855084032,"uniform":4.859542917762095,"feasible_rate":0.0439453125,"uniform_feasible_rate":0.0234375,"gap":0.02732387353746546,"exact_ms":0.8459158707410097,"qaoa_ms":38.754125125706196,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":11,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":5.046864855084032,"uniform":5.070688514782901,"feasible_rate":0.03515625,"uniform_feasible_rate":0.0244140625,"gap":0.02732387353746546,"exact_ms":0.3459160216152668,"qaoa_ms":36.741500021889806,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_12","choices":12,"budget_minutes":240,"seed":23,"exact":5.074188728621498,"greedy":5.074188728621498,"qaoa":4.286062678062677,"uniform":5.00592061819459,"feasible_rate":0.08203125,"uniform_feasible_rate":0.03125,"gap":0.7881260505588203,"exact_ms":0.3245000261813402,"qaoa_ms":36.503749899566174,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":7,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":5.724745317473793,"uniform":7.397927130201102,"feasible_rate":0.0556640625,"uniform_feasible_rate":0.025390625,"gap":1.732767495314529,"exact_ms":3.746833186596632,"qaoa_ms":218.91525015234947,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":11,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":6.5464964209291905,"uniform":6.942422346855117,"feasible_rate":0.01953125,"uniform_feasible_rate":0.0205078125,"gap":0.911016391859131,"exact_ms":3.8906249683350325,"qaoa_ms":199.8574580065906,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_15","choices":15,"budget_minutes":360,"seed":23,"exact":7.4575128127883215,"greedy":7.4575128127883215,"qaoa":5.400124928344106,"uniform":6.720006512006511,"feasible_rate":0.0400390625,"uniform_feasible_rate":0.013671875,"gap":2.057387884444215,"exact_ms":3.4437079448252916,"qaoa_ms":210.12112498283386,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":7,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":9.739391439610616,"uniform":12.263124772943948,"feasible_rate":0.0087890625,"uniform_feasible_rate":0.0107421875,"gap":3.0755168091168077,"exact_ms":29.978665988892317,"qaoa_ms":2166.687416145578,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":11,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":8.361801607919835,"uniform":12.222445570664746,"feasible_rate":0.0029296875,"uniform_feasible_rate":0.0078125,"gap":4.453106640807588,"exact_ms":28.78170902840793,"qaoa_ms":2101.9306250382215,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"mixed_gaps_18","choices":18,"budget_minutes":600,"seed":23,"exact":12.814908248727424,"greedy":10.936352195627705,"qaoa":10.918571754171753,"uniform":10.506143654417626,"feasible_rate":0.009765625,"uniform_feasible_rate":0.0126953125,"gap":1.8963364945556709,"exact_ms":27.77029201388359,"qaoa_ms":2133.5973341483623,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":7,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":4.885948717948717,"uniform":4.919081958022355,"feasible_rate":0.041015625,"uniform_feasible_rate":0.0234375,"gap":1.33048433048433,"exact_ms":0.803499948233366,"qaoa_ms":38.1330840755254,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":11,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":6.216433048433047,"uniform":6.216433048433047,"feasible_rate":0.033203125,"uniform_feasible_rate":0.0244140625,"gap":0.0,"exact_ms":0.34825014881789684,"qaoa_ms":39.76358310319483,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_12","choices":12,"budget_minutes":240,"seed":23,"exact":6.216433048433047,"greedy":6.216433048433047,"qaoa":5.3731282051282045,"uniform":4.885948717948717,"feasible_rate":0.0419921875,"uniform_feasible_rate":0.03125,"gap":0.8433048433048427,"exact_ms":0.5900000687688589,"qaoa_ms":39.06679106876254,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":7,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":5.485939100879498,"uniform":5.570780626780627,"feasible_rate":0.015625,"uniform_feasible_rate":0.025390625,"gap":1.9083401013997028,"exact_ms":3.326250007376075,"qaoa_ms":205.34425019286573,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":11,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":5.3731282051282045,"uniform":6.216433048433047,"feasible_rate":0.029296875,"uniform_feasible_rate":0.0205078125,"gap":2.021150997150996,"exact_ms":3.1799999997019768,"qaoa_ms":203.53570906445384,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_15","choices":15,"budget_minutes":360,"seed":23,"exact":7.3942792022792005,"greedy":7.235806267806266,"qaoa":6.7036125356125345,"uniform":6.7036125356125345,"feasible_rate":0.0302734375,"uniform_feasible_rate":0.013671875,"gap":0.6906666666666661,"exact_ms":3.8030000869184732,"qaoa_ms":208.2504581194371,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":7,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":4.904729344729344,"uniform":6.840328205128205,"feasible_rate":0.0107421875,"uniform_feasible_rate":0.0107421875,"gap":3.798276923076922,"exact_ms":32.20049990341067,"qaoa_ms":2143.3318748604506,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":11,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":6.3090052489575665,"uniform":6.819988603988603,"feasible_rate":0.0087890625,"uniform_feasible_rate":0.0078125,"gap":2.3940010188486998,"exact_ms":27.9897921718657,"qaoa_ms":2189.1555418260396,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false},{"case":"weak_derivatives_18","choices":18,"budget_minutes":600,"seed":23,"exact":8.703006267806266,"greedy":8.703006267806266,"qaoa":5.515578347578347,"uniform":7.926472934472935,"feasible_rate":0.013671875,"uniform_feasible_rate":0.0126953125,"gap":3.187427920227919,"exact_ms":30.48820816911757,"qaoa_ms":2430.2996250335127,"evaluations":30,"code_commit":"598bfa309cc373325276c9e3e5b749b7c5eed0e6","working_tree_dirty":false}]},"example":{"schema_version":3,"created_utc":"2026-11-06T17:15:16.498900+00:00","code_commit":"3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3","working_tree_dirty":false,"profile":{"schema_version":3,"profile":{"schema_version":2,"profile_source":"diagnostic","request":{"budget_minutes":600,"proficiency":{"T1":0.5,"T2":0.0,"T3":0.0},"requested_solver":"exact","block_minutes":15,"focus_minutes":45,"break_minutes":15},"topic_weights":{"T1":0.8333333333333333,"T2":0.6666666666666666,"T3":0.8333333333333333}},"sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T1-practice","topic_id":"T1","kind":"practice","start":660,"minutes":45,"requires":["d1-T1-review"]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T1-review","topic_id":"T1","kind":"review","start":3450,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T1-practice","topic_id":"T1","kind":"practice","start":3570,"minutes":45,"requires":["d1-T1-review","d2-T1-review","d3-T1-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"windows":[{"start":540,"end":780},{"start":1980,"end":2220},{"start":3420,"end":3660}],"deadlines":{"T1":2220,"T2":3660,"T3":3660}},"execution_mode":"live","reference":"https://quantum.cloud.ibm.com/docs/en/tutorials/quantum-approximate-optimization-algorithm","versions":{"qiskit":"2.6.2","numpy":"2.4.6","scipy":"1.17.1"},"exact":{"allocations":{"T1":90,"T2":180,"T3":180},"selected_sessions":[{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T2-practice","topic_id":"T2","kind":"practice","start":2100,"minutes":45,"requires":["d1-T2-review","d2-T2-review"]},{"session_id":"d2-T1-practice","topic_id":"T1","kind":"practice","start":2160,"minutes":45,"requires":["d1-T1-review","d2-T1-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"used_minutes":450,"break_minutes":105,"unused_minutes":45,"utility":12.814908248727424,"feasible":true,"actual_solver":"exact","execution_mode":"live","elapsed_ms":50.5322499666363},"greedy":{"allocations":{"T1":0,"T2":180,"T3":180},"selected_sessions":[{"session_id":"d1-T2-review","topic_id":"T2","kind":"review","start":570,"minutes":45,"requires":[]},{"session_id":"d1-T2-practice","topic_id":"T2","kind":"practice","start":690,"minutes":45,"requires":["d1-T2-review"]},{"session_id":"d2-T3-review","topic_id":"T3","kind":"review","start":2010,"minutes":45,"requires":[]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]},{"session_id":"d3-T2-practice","topic_id":"T2","kind":"practice","start":3600,"minutes":45,"requires":["d1-T2-review","d2-T2-review","d3-T2-review"]}],"used_minutes":360,"break_minutes":75,"unused_minutes":165,"utility":10.936352195627705,"feasible":true,"actual_solver":"greedy","execution_mode":"live","elapsed_ms":40.299874963238835},"uniform":{"raw_best_utility":12.263124772943948,"feasible_shot_rate":0.0107421875},"quantum":{"qubits":18,"p":1,"seed":7,"shots":1024,"objective_evaluations":30,"initial_params":[1.9637953256775056,2.818680285825393],"initial_selection_probability":0.35,"cost_scale":23.359088579456937,"parameter_order":["beta","gamma"],"params":[3.0894641741197972,2.7861868756245123],"optimizer_success":false,"expected_energy":82.42976213173037,"best_sample":{"allocations":{"T1":90,"T2":90,"T3":225},"selected_sessions":[{"session_id":"d1-T1-review","topic_id":"T1","kind":"review","start":540,"minutes":45,"requires":[]},{"session_id":"d1-T3-review","topic_id":"T3","kind":"review","start":600,"minutes":45,"requires":[]},{"session_id":"d1-T3-practice","topic_id":"T3","kind":"practice","start":720,"minutes":45,"requires":["d1-T3-review"]},{"session_id":"d2-T2-review","topic_id":"T2","kind":"review","start":1980,"minutes":45,"requires":[]},{"session_id":"d2-T1-review","topic_id":"T1","kind":"review","start":2040,"minutes":45,"requires":[]},{"session_id":"d2-T3-practice","topic_id":"T3","kind":"practice","start":2130,"minutes":45,"requires":["d1-T3-review","d2-T3-review"]},{"session_id":"d3-T3-review","topic_id":"T3","kind":"review","start":3420,"minutes":45,"requires":[]},{"session_id":"d3-T2-review","topic_id":"T2","kind":"review","start":3480,"minutes":45,"requires":[]},{"session_id":"d3-T3-practice","topic_id":"T3","kind":"practice","start":3540,"minutes":45,"requires":["d1-T3-review","d2-T3-review","d3-T3-review"]}],"used_minutes":405,"break_minutes":90,"unused_minutes":105,"utility":10.300167326331707,"feasible":true,"actual_solver":"qaoa","execution_mode":"live","elapsed_ms":0},"raw_best_utility":10.300167326331707,"feasible_shot_rate":0.05078125,"elapsed_ms":3167.955958051607,"counts":{"000000000000010110":1,"000000000000011101":1,"000000000000100011":1,"000000000000110000":1,"000000000000110001":1,"000000000000111010":1,"000000000001100010":1,"000000000010000001":1,"000000000010101100":1,"000000000011001000":1,"000000000011100001":1,"000000000100000001":1,"000000000100000010":1,"000000000100001100":1,"000000000100001110":1,"000000000100010010":1,"000000000101100010":1,"000000000101110010":1,"000000000110110001":1,"000000000111101110":1,"000000001001100000":1,"000000001001101001":1,"000000001010000000":1,"000000001010001110":1,"000000001010011000":1,"000000001011011000":1,"000000001101001001":1,"000000001101001101":1,"000000001101111100":1,"000000010000000001":1,"000000010000000010":1,"000000010000001000":2,"000000010000001110":1,"000000010000011100":1,"000000010000100001":1,"000000010000101000":1,"000000010001000011":1,"000000010001000100":1,"000000010001000101":1,"000000010001010001":1,"000000010010110010":1,"000000010011110101":1,"000000010100001001":1,"000000010100001100":1,"000000010100110000":1,"000000010110000000":1,"000000010110100001":1,"000000011000010010":1,"000000011000100001":1,"000000011010000110":1,"000000011011001001":1,"000000100000000001":1,"000000100000010100":1,"000000100000010101":1,"000000100001000101":1,"000000100010001111":1,"000000100010010100":1,"000000100100001010":2,"000000100100001100":1,"000000100100010010":1,"000000100110010011":1,"000000101000010101":1,"000000101001000010":1,"000000101001100111":1,"000000101010010101":1,"000000101100001100":1,"000000101101111001":1,"000000110000000100":1,"000000110000001000":1,"000000110000010010":1,"000000110000011010":1,"000000110000101110":1,"000000110000110100":1,"000000110010000000":1,"000000110010101010":1,"000000110100000011":1,"000000110100010011":1,"000000111001010110":1,"000000111001100010":1,"000000111011011100":1,"000000111110010000":1,"000001000000000010":1,"000001000000010101":1,"000001000000100001":1,"000001000000100010":1,"000001000010000000":1,"000001000010100011":1,"000001000010110001":1,"000001000011000101":1,"000001000101000101":1,"000001000101100000":1,"000001001000000011":1,"000001001110011000":1,"000001001110100010":1,"000001001111010010":1,"000001010000010011":1,"000001010000110001":1,"000001010100110000":1,"000001010101001001":1,"000001010101100010":1,"000001010101110011":1,"000001011000000000":1,"000001011110000001":1,"000001011110001100":1,"000001100000000110":1,"000001100000001100":1,"000001100010000010":1,"000001101000000000":1,"000001101000101001":1,"000001110111000010":1,"000001111000010001":1,"000001111010000000":2,"000010000000000000":1,"000010000000011001":1,"000010000000100010":1,"000010000000111000":1,"000010000100000101":1,"000010000100001010":1,"000010000100010010":1,"000010000101010101":1,"000010000101011000":1,"000010000101011110":1,"000010000110010011":1,"000010000110100000":1,"000010000110111101":1,"000010001000110010":1,"000010001001010011":1,"000010001001100011":1,"000010001001100111":1,"000010001010000100":1,"000010001010100100":1,"000010001011011100":1,"000010001100000100":1,"000010001100100110":1,"000010001100110000":1,"000010001101101000":1,"000010010000000001":1,"000010010000001101":1,"000010010000101100":1,"000010010001001110":1,"000010010010000001":1,"000010010010100000":1,"000010010011000101":1,"000010010011011001":1,"000010010100100010":1,"000010010100110110":1,"000010010110110101":1,"000010011000000001":1,"000010011000001101":1,"000010011010001100":1,"000010011100000000":1,"000010011100001000":1,"000010011101100000":1,"000010100001101110":1,"000010100010001000":1,"000010100100110010":1,"000010101000110001":1,"000010101001110001":1,"000010101010010010":1,"000010101100001010":1,"000010110000011001":1,"000010110001100001":1,"000010110010000011":1,"000010110011111100":1,"000010110101001001":1,"000010111000100000":1,"000011000001000011":1,"000011000001110010":1,"000011000010010010":1,"000011000011011000":1,"000011000100001111":1,"000011001000001100":1,"000011001000100110":1,"000011001001010001":1,"000011001001100000":1,"000011001001101001":1,"000011001101101110":1,"000011010111000001":1,"000011011101010001":1,"000011011110000110":1,"000011100000010010":1,"000011100000010100":1,"000011100110010000":1,"000011101000001101":1,"000011101000110100":1,"000011101100100101":1,"000011110010011011":1,"000011110010100100":1,"000011111001001100":1,"000100000000100100":1,"000100000000101010":1,"000100000000101011":1,"000100000000110000":1,"000100000000110110":1,"000100000001000000":2,"000100000001000010":1,"000100000010001111":1,"000100000010100101":1,"000100000010111100":1,"000100000100000100":1,"000100000101011110":1,"000100000101111001":1,"000100000110010010":1,"000100001000000100":1,"000100001000000110":1,"000100001000000111":1,"000100001001110001":1,"000100001011011000":1,"000100001011011100":1,"000100001110010110":1,"000100010000000001":1,"000100010000000100":1,"000100010000001000":1,"000100010000010111":1,"000100010001010100":1,"000100010010001101":1,"000100010010010110":1,"000100010010100101":1,"000100010100000001":1,"000100010101010011":1,"000100010101100100":1,"000100010110000101":1,"000100010110011111":1,"000100010111001100":1,"000100011010000100":1,"000100011010010111":1,"000100011100011010":1,"000100100000110101":1,"000100100010001100":1,"000100100100000001":1,"000100101000001101":1,"000100101001000001":1,"000100101011001111":1,"000100101100110000":1,"000100101101101001":1,"000100101111010000":1,"000100101111101000":1,"000100110000000101":1,"000100110010000100":1,"000100110100000010":1,"000100111011000000":1,"000100111100100100":1,"000101000001010010":1,"000101000101000000":1,"000101000111110110":1,"000101001000101100":1,"000101010000001001":1,"000101010000111110":1,"000101011001000100":1,"000101100010000101":1,"000101101011001001":1,"000101101011110100":1,"000110000000010111":1,"000110000001000000":1,"000110000001010100":1,"000110000010000100":1,"000110000010011011":1,"000110000100000000":1,"000110000100000001":1,"000110000101000111":1,"000110000111010100":1,"000110001001101101":1,"000110001110000000":1,"000110001110010011":1,"000110001110010100":1,"000110010000000001":1,"000110010000010000":1,"000110010000011001":1,"000110010010110010":1,"000110010101010100":1,"000110010111010010":1,"000110011001001010":1,"000110011110011101":1,"000110100000000010":1,"000110100011011110":1,"000110110000001100":1,"000110110100111000":1,"000110111010110100":1,"000110111100011001":1,"000110111110101001":1,"000111000000110000":1,"000111000010010110":1,"000111000010011001":1,"000111000110000001":1,"000111001001010010":1,"000111001011100011":1,"000111001110110001":1,"000111010100101001":1,"000111011000010110":1,"000111100011110001":1,"000111110000001000":1,"001000000000101000":1,"001000000000111000":2,"001000000001000000":1,"001000000010000000":1,"001000000010001000":1,"001000000010101101":1,"001000000011101010":1,"001000000100000010":1,"001000000100110010":1,"001000000111010110":1,"001000001000001010":1,"001000001000101000":1,"001000001001110010":1,"001000001010001000":1,"001000001010011010":1,"001000001011100000":1,"001000001111100101":1,"001000010000000010":2,"001000010010100110":1,"001000010011011110":1,"001000011001010001":1,"001000011010010001":1,"001000011011000000":1,"001000011111000000":1,"001000100000101001":1,"001000100001100011":1,"001000100001100100":1,"001000100011100010":1,"001000100100010010":1,"001000100100110110":1,"001000100101010001":1,"001000101110010101":1,"001000101110100010":1,"001000110000000010":1,"001000110000011000":1,"001000110001110100":1,"001000110011000010":1,"001000110011101000":1,"001000110110000001":1,"001000110110010011":1,"001000111011010000":1,"001000111100000111":1,"001001000000001101":1,"001001000000010100":1,"001001000001010001":1,"001001000001010011":1,"001001000011101000":1,"001001000110000101":1,"001001001000000100":1,"001001001001010000":1,"001001001001100100":1,"001001001010011000":1,"001001010011101110":1,"001001010110000000":1,"001001011100000000":1,"001001011110000010":1,"001001100001010100":1,"001001100010001101":1,"001001100100100000":1,"001001100101010001":1,"001001100110100011":1,"001001101000000010":1,"001001101010001001":1,"001001110000001010":1,"001001110001000011":1,"001001110010000010":1,"001001110010101001":1,"001001111000000000":1,"001010000000000110":1,"001010000000100000":1,"001010000010000001":1,"001010001000011001":1,"001010001001000100":1,"001010001110000010":1,"001010010011100000":1,"001010011010111001":1,"001010100010100110":1,"001010100110000001":1,"001010101110001000":1,"001010110010110001":1,"001011000101001110":1,"001011001100000011":1,"001011101010000010":1,"001011101111010100":1,"001011111000100100":1,"001100000000000111":1,"001100000000001001":1,"001100000000011100":1,"001100000000100001":1,"001100000001000001":1,"001100000001100000":1,"001100000010010110":1,"001100000011001000":1,"001100000100001010":1,"001100000101110011":1,"001100000110100001":1,"001100000111010001":1,"001100001000000101":1,"001100001001110110":1,"001100001010000001":1,"001100001100000010":1,"001100001100100110":1,"001100001101000010":1,"001100010000010111":1,"001100010000100101":1,"001100010001010000":1,"001100010010010100":1,"001100010010010110":1,"001100100010010110":1,"001100100110100010":1,"001100100110110011":1,"001100100111000000":1,"001100100111110111":1,"001100110001000111":1,"001101000000001000":1,"001101000011000000":1,"001101000101001010":1,"001101001000000000":1,"001101001000111000":1,"001101010001110100":1,"001101010101100101":1,"001101101000010000":1,"001101101000011000":1,"001101101011101110":1,"001101101101101010":1,"001101110101101010":1,"001101111000100000":1,"001110000001000010":1,"001110000001100000":1,"001110000010000000":1,"001110000010100011":1,"001110000100110010":1,"001110001010111001":1,"001110010100010111":1,"001110010101001000":1,"001110011000010001":1,"001110011000111100":1,"001110110001000010":1,"001111000111001000":1,"001111010100010011":1,"001111100001000000":1,"001111100110001000":1,"001111101100011000":1,"001111110100000000":1,"010000000000000110":1,"010000000000010001":1,"010000000010000010":1,"010000000010111110":1,"010000000011011011":1,"010000000100000000":1,"010000000100010110":1,"010000000101000000":1,"010000000111110000":1,"010000001000001111":1,"010000001001010001":1,"010000001001101111":1,"010000001010000000":1,"010000001100111000":1,"010000001101000001":1,"010000001101010100":1,"010000010001000001":1,"010000010010000100":1,"010000010100000001":1,"010000010111000110":1,"010000011000000010":1,"010000011000010001":1,"010000011000011001":1,"010000011010010010":1,"010000011101000001":1,"010000011101010000":1,"010000011101111001":1,"010000100000000010":1,"010000100000000100":1,"010000100100100100":1,"010000100101010100":1,"010000100101101000":1,"010000100110000110":1,"010000101000011000":1,"010000101001000010":1,"010000101001100001":1,"010000101011010010":1,"010000101101111000":1,"010000110000010110":1,"010000110001001010":1,"010000110001010101":1,"010000110010100000":1,"010001000000101000":1,"010001000001111010":1,"010001000010000100":1,"010001000010010110":1,"010001000010110110":1,"010001000010111010":1,"010001000011000000":1,"010001000011100101":1,"010001000100001001":1,"010001000110000001":1,"010001000110011000":1,"010001001000001001":1,"010001001001000000":1,"010001001001010010":1,"010001001001010100":1,"010001001101000000":1,"010001001101010010":1,"010001010010011101":1,"010001010011000110":1,"010001010100011011":1,"010001010101000001":1,"010001011000110101":1,"010001011101010111":1,"010001100100001010":1,"010001111010010010":1,"010010000001100111":1,"010010000010110000":1,"010010000100100001":1,"010010001000001010":1,"010010001100011011":1,"010010010000000001":1,"010010010001000000":1,"010010011000000000":1,"010010011001001010":1,"010010101011001001":1,"010010110000000100":1,"010010110000110010":1,"010010110000111110":1,"010010110010000000":1,"010010110010010001":1,"010010110100100001":1,"010010111000110010":1,"010011000000100010":1,"010011000001000000":1,"010011000011000100":1,"010011000011011011":1,"010011001001110001":1,"010011010001000011":1,"010011011010000011":1,"010011011011001000":1,"010011100010000011":1,"010011100100000000":1,"010011100110000100":1,"010011101010011010":1,"010011110000001000":1,"010011110000100110":1,"010100000000100110":1,"010100000000101000":1,"010100000000111000":1,"010100000001001000":1,"010100000001001010":1,"010100000010110101":1,"010100000010111000":1,"010100000011000010":1,"010100000100000000":1,"010100000110011001":1,"010100000110011010":1,"010100001000000000":1,"010100001000001010":1,"010100001001000010":1,"010100001111000110":1,"010100101001110000":1,"010100101010000100":1,"010100110011110001":1,"010100111000001010":1,"010100111010100110":1,"010101000000000001":1,"010101000001000101":1,"010101000010000011":1,"010101000010001111":1,"010101000011100000":1,"010101001011000101":1,"010101010000001001":1,"010101010100010110":1,"010101010111100000":1,"010101011001010000":1,"010101100100000010":1,"010101100100000100":1,"010101100110000011":1,"010101101000000100":1,"010101101111110000":1,"010101110010100001":1,"010110000000000010":1,"010110000000000011":1,"010110001100010000":1,"010110001110000000":1,"010110010000000000":1,"010110010010101000":1,"010110101000101010":1,"010110110000110000":1,"010110110010100010":1,"010110111000100001":1,"010111001100000100":1,"010111010100001001":1,"010111100000000001":1,"010111101100001101":1,"010111101100101000":1,"010111110011110100":1,"011000000000001001":1,"011000000001000000":1,"011000000010000110":1,"011000000010011111":1,"011000000100000000":1,"011000000100011001":1,"011000000101000000":1,"011000000110000011":1,"011000001001111011":1,"011000001100101010":1,"011000001101000000":1,"011000001101101010":1,"011000001110001000":1,"011000010001000000":1,"011000010010101011":1,"011000010110010100":1,"011000011000100000":1,"011000011001010000":1,"011000011001011100":1,"011000100100000101":1,"011000101101000100":1,"011000110000010100":1,"011000110000100010":1,"011000110001010011":1,"011000110001101011":1,"011001000001000111":1,"011001000100110100":1,"011001000101001100":1,"011001001101001001":1,"011001010101000011":1,"011001100001100000":1,"011001100011010011":1,"011001101001100000":1,"011001110100001001":1,"011001110101011011":1,"011010000000010100":1,"011010000010000111":1,"011010000010100001":1,"011010001010000010":1,"011010001010111001":1,"011010010000101010":1,"011010010001001110":1,"011010010111001100":1,"011010011100100100":1,"011010100000000010":1,"011010100001011011":1,"011011000001100110":1,"011011000101000010":1,"011011001111001000":1,"011011010100111100":1,"011011011010011001":1,"011011100001000001":1,"011011101100000010":1,"011011110000100000":1,"011100000000000110":1,"011100000000110010":1,"011100000100000001":1,"011100001000100000":1,"011100010000000001":1,"011100010000110000":1,"011100010001000000":1,"011100010010010110":1,"011100010100001001":1,"011100010110000111":1,"011100100000010001":1,"011100100000011001":1,"011100100100000011":1,"011100101100001110":1,"011101000000010001":1,"011101000001100101":1,"011101000010100011":1,"011101000011001110":1,"011101000100011010":1,"011101001001110110":1,"011101010001010001":1,"011101011000110001":1,"011101100100000001":1,"011101100100100111":1,"011101110001011010":1,"011101111100001101":1,"011110000000010001":1,"011110000000111000":1,"011110001000000100":1,"011110001001100101":1,"011110010000101001":1,"011110101010010000":1,"011111100111111100":1,"100000000001110010":1,"100000000010000010":1,"100000000010001101":1,"100000000010100001":1,"100000000010101101":1,"100000000100000000":1,"100000000100100000":1,"100000000101001111":1,"100000000110000001":1,"100000000110100111":1,"100000000111000011":1,"100000001000001000":1,"100000001001000011":1,"100000001001001001":1,"100000001010011110":1,"100000001110011010":1,"100000010000010000":1,"100000010000011100":1,"100000010000100101":1,"100000010010100010":1,"100000010010101100":1,"100000010110101000":1,"100000011010110000":1,"100000011011000001":1,"100000011011000011":1,"100000011111000100":1,"100000100000000000":1,"100000100000000100":1,"100000100000001101":1,"100000100000010110":1,"100000100010000110":1,"100000100011100100":1,"100000100101111101":1,"100000101000000000":1,"100000101011010011":1,"100000101101100110":1,"100000101111010000":1,"100001000000000010":1,"100001000000000100":1,"100001000000100110":1,"100001000001010100":1,"100001000100000000":1,"100001000110001110":1,"100001001001000000":1,"100001001001000011":1,"100001010011001011":1,"100001010011001101":1,"100001010100100000":1,"100001100010000000":1,"100001100010000011":1,"100001100100000101":1,"100001100101001000":1,"100001101000010000":1,"100001101101000111":1,"100001110011000001":1,"100001111101110000":1,"100001111111001001":1,"100010000000000010":1,"100010000000000110":1,"100010000000100011":1,"100010000100111000":1,"100010001010010111":1,"100010001011000001":1,"100010001100010111":1,"100010001110010000":1,"100010010000000001":1,"100010010000011000":1,"100010010001000100":1,"100010010111010101":1,"100010011010000010":1,"100010100000000000":1,"100010100010011101":1,"100010100100000110":1,"100010101100000000":1,"100010101100100000":1,"100010110000110001":1,"100010110001000000":1,"100010111101011010":1,"100011000000110000":1,"100011000101101001":1,"100011000111000100":1,"100011001100000100":1,"100011010100000010":1,"100011100000010110":1,"100011100111001001":1,"100011110000001000":1,"100100000000000001":1,"100100000000101001":1,"100100000100000010":1,"100100001000000001":1,"100100001000000010":1,"100100001000000110":1,"100100001000100000":1,"100100001001011100":1,"100100001011101100":1,"100100001101000000":1,"100100001110000000":1,"100100010000101101":1,"100100010010100110":1,"100100010011001010":1,"100100010100000110":1,"100100010100001110":1,"100100011000000000":1,"100100011000000100":1,"100100011000001110":1,"100100011011001000":1,"100100011100010100":1,"100100011100100000":1,"100100100010000000":1,"100100100011001001":1,"100100100100001100":1,"100100100101000001":1,"100100100110000100":1,"100100100110100000":1,"100100110000001111":1,"100100110011000000":1,"100100110111001101":1,"100100111010100000":1,"100101000001000011":1,"100101000010011001":1,"100101000010111000":1,"100101000100000001":1,"100101000110001001":1,"100101000110011000":1,"100101001010110011":1,"100101010001000001":1,"100101010010001010":1,"100101100100100000":1,"100101101001000100":1,"100101101110111001":1,"100110000100001000":1,"100110001010000100":1,"100110010010010010":1,"100110011001001010":1,"100110011100010101":1,"100110110011010011":1,"100111000010001101":1,"100111000011110100":1,"100111000110001000":1,"100111101001000010":1,"101000000000000000":1,"101000000000000001":1,"101000000000001000":1,"101000000010000001":1,"101000000010000010":1,"101000000100001010":1,"101000000100110000":1,"101000000101000000":1,"101000000101010010":1,"101000000110000000":1,"101000000110111011":1,"101000001000000001":1,"101000001000011010":1,"101000001000101000":1,"101000001010011100":1,"101000001110101100":1,"101000010000010100":1,"101000010001000010":1,"101000010001011001":1,"101000010100000000":1,"101000011100001000":1,"101000100000011110":1,"101000100001100000":1,"101000100010100011":1,"101000100011001000":1,"101000101010010000":1,"101000101011010001":1,"101000101011100011":1,"101000110000110000":1,"101000110010100011":1,"101000110011001010":1,"101000110100001000":1,"101000110101101000":1,"101000111000010000":1,"101000111000110010":1,"101001000000001100":1,"101001000101000100":1,"101001010000111000":1,"101001011000100000":1,"101001011100000100":1,"101001100010001001":1,"101001100110001000":1,"101001101001010000":1,"101001110001010110":1,"101001111110101000":1,"101010000000010111":1,"101010000000100000":1,"101010000001000000":1,"101010000101000100":1,"101010001001100100":1,"101010001111000001":1,"101010101101001000":1,"101011000011000110":1,"101011001010000001":1,"101011001101001010":1,"101011010000100000":1,"101011100000010000":1,"101011100011101010":1,"101100000000000101":1,"101100000000110001":1,"101100000100100100":1,"101100000100111000":1,"101100001101000000":1,"101100010000001110":1,"101100110100001011":1,"101100110111111100":1,"101100111110000010":1,"101101000000000010":1,"101101000110101000":1,"101101001001010000":1,"101101100000100110":1,"101101100110011100":1,"101101101100100001":1,"101101110011111000":1,"101101111000000011":1,"101110000011010010":1,"101110000100001100":1,"101110010011000001":1,"101110010011110111":1,"101110100010100000":1,"101110101000101000":1,"101110101010000010":1,"101110110001011001":1,"101111010001000000":1,"101111110010000000":1,"101111111000011011":1,"110000000000000001":1,"110000000000010000":1,"110000000001110100":1,"110000000010010010":1,"110000000100101001":1,"110000000111001100":1,"110000000111101100":1,"110000001000001011":1,"110000001000101100":1,"110000001100000111":1,"110000001111000001":1,"110000010010011000":1,"110000010011100010":1,"110000010100000100":1,"110000011000011110":1,"110000011100110011":1,"110000100000110110":1,"110000100010000011":1,"110000101100010101":1,"110000101100100001":1,"110000101100110001":1,"110000110000001100":1,"110000110001100010":1,"110000110110001011":1,"110001000110001001":1,"110001001010000011":1,"110001010010011000":1,"110001010101100010":1,"110001010111100010":1,"110001100101100000":1,"110001101101110000":1,"110010000001100000":1,"110010000011111000":1,"110010000100110011":1,"110010000101001011":1,"110010000110100100":1,"110010000111100000":1,"110010001001001010":1,"110010001100011101":1,"110010010000110001":1,"110010010001001001":1,"110010010111000000":1,"110010010111000011":1,"110010011111100011":1,"110010100010100000":1,"110010100110100101":1,"110010101000000000":1,"110010101000101011":1,"110010101100000001":1,"110010110000110110":1,"110010110001000100":1,"110011000001011000":1,"110011000110000000":1,"110011001100100000":1,"110011001101010010":1,"110100000000000000":1,"110100000010000100":1,"110100001001000100":1,"110100001110101011":1,"110100010100111100":1,"110100011000000000":1,"110100011110001010":1,"110100011111000000":1,"110100100000000000":1,"110100100011110000":1,"110100101110010001":1,"110100111010001001":1,"110101000111000101":1,"110101001000001010":1,"110101001110111000":1,"110101100100100010":1,"110101100111010101":1,"110101110000111111":1,"110110000001011100":1,"110110000011100101":1,"110110000101000110":1,"110110001000000000":1,"110110001101000110":1,"110110010000000001":1,"110110010000110010":1,"110110011001111001":1,"110110011101000101":1,"110111100100001000":1,"111000000000110000":2,"111000000010000010":1,"111000000011100100":1,"111000000100011010":1,"111000000101100000":1,"111000000110010110":1,"111000001000110011":1,"111000010100101000":1,"111000100000101000":1,"111000100011010001":1,"111000101000010000":1,"111000110001001001":1,"111000110010011001":1,"111000110101000001":1,"111001000001100101":1,"111001001000010010":1,"111001100001100001":1,"111010000010000000":1,"111010001000100100":1,"111010010000101100":1,"111010011001001010":1,"111010100000000110":1,"111010110000010100":1,"111100001001011100":1,"111100010011011110":1,"111100100000000000":1,"111100100100001001":1,"111100100111000110":1,"111100110011001010":1,"111101000000000000":1,"111101001000100000":1,"111110000000010000":1,"111110000001000010":1,"111110010011101001":1,"111110100000000001":1,"111111010001000111":1},"trace":[{"params":[1.9637953256775056,2.818680285825393],"energy":911.3575426253253},{"params":[2.9637953256775056,2.818680285825393],"energy":86.13147574401869},{"params":[2.9637953256775056,3.818680285825393],"energy":86.44602728495465},{"params":[3.963795253032173,2.8182991157064294],"energy":358.34040581041825},{"params":[2.4637956595018657,2.8181025105675226],"energy":190.0528967491549},{"params":[3.2137950393779606,2.818301934546168],"energy":84.44979820019977},{"params":[3.313685870948883,2.813630557011682],"energy":89.08584595364874},{"params":[3.2114593506107174,2.768356518760707],"energy":84.18297703827207},{"params":[3.1116895927061554,2.76157452733745],"energy":82.94469059954193},{"params":[3.018689714576115,2.724818267084296],"energy":86.82669284327025},{"params":[3.10461258819281,2.8110711535570028],"energy":82.61726784309224},{"params":[3.081214875001443,2.8198772319926366],"energy":82.75138428394561},{"params":[3.106373803879937,2.815750696195276],"energy":82.64432246774102},{"params":[3.108700578285012,2.8019449091483724],"energy":82.5677671029762},{"params":[3.1079309869681495,2.791974566666541],"energy":82.5016365203735},{"params":[3.1012135423371032,2.7731364155463054],"energy":82.60244138588041},{"params":[3.1176726341136884,2.789716179876858],"energy":82.56683536103962},{"params":[3.103917753126492,2.7889922933503986],"energy":82.46867361672795},{"params":[3.099035677954101,2.7879127832711847],"energy":82.44655130584967},{"params":[3.0970946337821488,2.783304926120347],"energy":82.44547754398957},{"params":[3.094753098943161,2.7841808339909626],"energy":82.43721157100049},{"params":[3.092391014733314,2.784999710163281],"energy":82.43203616875654},{"params":[3.089891032914282,2.7850092445489945],"energy":82.43088268096315},{"params":[3.0896727491906693,2.787499696760141],"energy":82.43118963444138},{"params":[3.088948479197459,2.7846751899263618],"energy":82.43181155250166},{"params":[3.0900580602255983,2.7845379676905826],"energy":82.43204268427101},{"params":[3.0899312492086106,2.786008435546587],"energy":82.42976381465891},{"params":[3.0904097698122945,2.7868865118712023],"energy":82.43023251896713},{"params":[3.0894641741197972,2.7861868756245123],"energy":82.42976213173037},{"params":[3.089373292755421,2.785953979558403],"energy":82.42980878710149}]},"absolute_utility_gap":2.514740922395717}}')
benchmark = DATA["benchmark"]
earlier = DATA["initial_benchmark"]
example = DATA["example"]
rows = benchmark["results"]
assert len(rows) == 27
assert len({r["case"] for r in rows}) == 9
assert {r["seed"] for r in rows} == {7, 11, 23}
assert all(r["exact"] > 0 for r in rows)
print("Loaded 27 recorded runs: nine fictional problems, with three seeds each.")
print("Current benchmark commit:", rows[0]["code_commit"])
print("Earlier benchmark commit:", earlier["results"][0]["code_commit"])
print("Settings:", benchmark["settings"])


Loaded 27 recorded runs: nine fictional problems, with three seeds each.
Current benchmark commit: 3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3
Earlier benchmark commit: 598bfa309cc373325276c9e3e5b749b7c5eed0e6
Settings: {'initial_state': 'budget-aware independent ry rotations', 'seeds': [7, 11, 23], 'p': 1, 'max_evaluations': 30, 'shots': 1024}


## 2. Why these comparisons matter

Exact search evaluates every combination and establishes the best possible score for each small instance. Greedy search adds the most useful allowed session at each step, while uniform random sampling shows what an unguided search can achieve with the same final sample count as QAOA.

Our Qiskit experiment uses one qubit per candidate session. It prepares a budget-informed initial state with RY rotations, applies a cost-dependent phase directly to amplitudes using NumPy, and uses Qiskit RX gates for mixing. SciPy COBYLA tunes two angles over at most 30 objective evaluations. Optimization uses exact statevector probabilities, followed by 1,024 sampled schedules from which the best valid one is selected. This is ideal local simulation rather than a hardware run or a fully compiled hardware cost circuit.

We distinguish three outcomes: whether a method finds a valid schedule, how its best valid schedule scores relative to the exact answer, and how long the recorded computation takes. A high valid-sample rate does not necessarily produce the highest-quality plan.


In [3]:
def print_table(headers, records):
    text = [[str(v) for v in headers]] + [[str(v) for v in row] for row in records]
    widths = [max(len(row[c]) for row in text) for c in range(len(headers))]
    for i, row in enumerate(text):
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))
        if i == 0:
            print("  ".join("-" * width for width in widths))

summary = []
for label, key in [("Exact reference", "exact"), ("Greedy", "greedy"),
                   ("Uniform random", "uniform"), ("QAOA simulation", "qaoa")]:
    valid = [r for r in rows if r[key] is not None]
    ratio = mean(r[key] / r["exact"] for r in valid)
    summary.append([label, f"{100*ratio:.1f}%", f"{len(valid)}/{len(rows)}"])
print_table(["Method", "Mean score / exact", "Runs returning a valid score"], summary)
matched = sum(r["qaoa"] is not None and abs(r["qaoa"] - r["exact"]) < 1e-9 for r in rows)
wins = sum(r["qaoa"] > r["uniform"] + 1e-9 for r in rows)
ties = sum(abs(r["qaoa"] - r["uniform"]) < 1e-9 for r in rows)
print(f"\nQAOA matched the exact score in {matched}/27 runs.")
print(f"Against random sampling: {wins} wins, {ties} ties, {27-wins-ties} losses.")
print("QAOA beat greedy in", sum(r["qaoa"] > r["greedy"] + 1e-9 for r in rows), "runs.")
assert matched == 4


Method           Mean score / exact  Runs returning a valid score
---------------  ------------------  ----------------------------
Exact reference  100.0%              27/27                       
Greedy           98.1%               27/27                       
Uniform random   86.8%               27/27                       
QAOA simulation  85.1%               27/27                       

QAOA matched the exact score in 4/27 runs.
Against random sampling: 12 wins, 2 ties, 13 losses.
QAOA beat greedy in 0 runs.


## 3. Results across the tested configurations

These are nine fictional problem instances, each repeated with three random seeds, rather than 27 independent students. Candidate count and budget change together across configurations, so the differences below do not isolate the effect of problem size. The quality percentage is the average of each run's score divided by that same instance's exact optimum.


In [4]:
records = []
for size in (12, 15, 18):
    group = [r for r in rows if r["choices"] == size]
    records.append([
        size, group[0]["budget_minutes"],
        f"{100*mean(r['qaoa']/r['exact'] for r in group):.1f}%",
        f"{100*mean(r['greedy']/r['exact'] for r in group):.1f}%",
        f"{median(r['exact_ms'] for r in group):.3f}",
        f"{median(r['qaoa_ms'] for r in group):.3f}",
    ])
print_table(["Choices", "Budget (min)", "QAOA / exact", "Greedy / exact",
             "Exact median ms", "QAOA median ms"], records)
print("\nRecorded local runtimes; these are not quantum hardware timings.")


Choices  Budget (min)  QAOA / exact  Greedy / exact  Exact median ms  QAOA median ms
-------  ------------  ------------  --------------  ---------------  --------------
12       240           93.5%         100.0%          0.436            41.775        
15       360           83.5%         99.3%           3.809            244.159       
18       600           78.3%         95.1%           37.939           2869.293      

Recorded local runtimes; these are not quantum hardware timings.


## 4. What changed when we revised the starting state?

The earlier variant began with a uniform superposition, which gives every session a 50% initial selection chance. The revised variant uses the time budget to favour fewer selected sessions. We compare matched problem-and-seed pairs from the earlier and revised benchmarks rather than changing the tested student profiles.

The share of sampled schedules obeying every rule rose from about 2.8% to 8.2%. The mean best schedule score also improved, but this does not isolate the benefit of QAOA's quantum evolution. A necessary next control is to sample the revised initial state without the QAOA steps, using the same sample budget.


In [5]:
key = lambda r: (r["case"], r["seed"])
old = {key(r): r for r in earlier["results"]}
assert set(old) == {key(r) for r in rows}
for r in rows:
    previous = old[key(r)]
    assert abs(r["exact"] - previous["exact"]) < 1e-9
    assert r["budget_minutes"] == previous["budget_minutes"]
comparison = []
for label, results in [("Earlier uniform-start QAOA", earlier["results"]),
                       ("Revised budget-informed QAOA", rows)]:
    comparison.append([label,
        f"{100*mean(r['feasible_rate'] for r in results):.2f}%",
        f"{100*mean(r['qaoa']/r['exact'] for r in results):.2f}%"])
print_table(["Variant", "Mean valid-sample rate", "Mean best score / exact"], comparison)
print("\nAll 27 case-and-seed pairs matched. Initialization-only control is still missing.")


Variant                       Mean valid-sample rate  Mean best score / exact
----------------------------  ----------------------  -----------------------
Earlier uniform-start QAOA    2.79%                   81.94%                 
Revised budget-informed QAOA  8.25%                   85.10%                 

All 27 case-and-seed pairs matched. Initialization-only control is still missing.


## 5. An example of the returned schedule

The following timetable comes from the saved `mixed_gaps` example with 18 choices and a 600-minute budget. It shows the exact reference plan so the abstract optimization score has a concrete interpretation. This is previously recorded output, not a live run of this notebook.


In [6]:
def show_schedule(result):
    records = []
    for item in result["selected_sessions"]:
        minute = item["start"] % 1440
        records.append([1 + item["start"] // 1440,
                        f"{minute//60:02d}:{minute%60:02d}",
                        item["topic_id"], item["kind"], item["minutes"]])
    print_table(["Day", "Start", "Topic", "Activity", "Minutes"], records)
    print(f"\nStudy: {result['used_minutes']} min; breaks: {result['break_minutes']} min; unused: {result['unused_minutes']} min")
    print(f"Planning score: {result['utility']:.3f}; valid: {result['feasible']}")

show_schedule(example["exact"])
assert example["exact"]["feasible"]
assert sum(example["exact"][k] for k in ("used_minutes", "break_minutes", "unused_minutes")) == 600


Day  Start  Topic  Activity  Minutes
---  -----  -----  --------  -------
1    10:00  T3     review    45     
1    12:00  T3     practice  45     
2    09:00  T2     review    45     
2    10:00  T1     review    45     
2    11:00  T2     practice  45     
2    12:00  T1     practice  45     
3    09:00  T3     review    45     
3    10:00  T2     review    45     
3    11:00  T3     practice  45     
3    12:00  T2     practice  45     

Study: 450 min; breaks: 105 min; unused: 45 min
Planning score: 12.815; valid: True


## 6. Run a fresh Qiskit comparison

Place this notebook at the root of the real cloned `Quantum-Lab` repository, alongside `app.py`. In the project's activated environment, install `requirements.txt` and `ipykernel` if they are not already installed, then select that environment as the notebook kernel. Set `RUN_LIVE = True` in the first cell and choose **Run All**.

This section imports the existing scheduler instead of maintaining a second implementation. It runs one of the 12-choice benchmark problems with seed 7, one QAOA layer, 30 objective evaluations and 1,024 final samples. Fresh runtime and results may differ across environments. The code checks that it is inside a real repository so the implementation can correctly record its commit. Any missing dependency or computation failure remains visible rather than falling back to saved output.


In [7]:
live_report = None
if RUN_LIVE:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / "src/optimizer/scheduler.py").is_file()
                 and (p / "data/demo/course.json").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Place this notebook inside the cloned Quantum-Lab repository.")
    actual_root = Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=ROOT, text=True).strip()).resolve()
    if actual_root != ROOT.resolve():
        raise RuntimeError("Use the real project clone, not a source folder inside another repository.")
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from src.optimizer.scheduler import ScheduleSnapshot, compare_schedule
    chosen = next(c for c in benchmark["cases"] if c["case"] == "weak_integrals_12")
    snapshot = ScheduleSnapshot.model_validate(chosen["profile"])
    live_report = compare_schedule(snapshot, seed=7, evaluations=30, shots=1024)
    print("FRESH LOCAL QISKIT RUN")
    print("Code:", live_report["code_commit"], "dirty:", live_report["working_tree_dirty"])
    print("Versions:", live_report["versions"])
    print("Exact score:", live_report["exact"]["utility"])
    print("QAOA score:", live_report["quantum"]["raw_best_utility"])
    print("Valid-sample rate:", live_report["quantum"]["feasible_shot_rate"])
    if live_report["quantum"]["best_sample"] is not None:
        show_schedule(live_report["quantum"]["best_sample"])
    else:
        print("No valid QAOA schedule appeared in this run.")
else:
    print("Fresh Qiskit execution is OFF. Set RUN_LIVE = True to run this section.")


FRESH LOCAL QISKIT RUN
Code: ae69d31018730050a25592d46b6d0fbebd18f03f dirty: True
Versions: {'qiskit': '2.6.2', 'numpy': '2.4.6', 'scipy': '1.17.1'}
Exact score: 4.049038255342889
QAOA score: 3.8612381442257346
Valid-sample rate: 0.1318359375
Day  Start  Topic  Activity  Minutes
---  -----  -----  --------  -------
1    09:30  T2     review    45     
1    11:30  T2     practice  45     
2    09:30  T3     review    45     
2    11:00  T2     practice  45     

Study: 180 min; breaks: 30 min; unused: 30 min
Planning score: 3.861; valid: True


### Optional: regenerate the complete 27-run benchmark

Enable both switches in the first cell to run the repository's original benchmark script. This takes longer than the single example and writes a new results file under `runtime/notebook/`; it does not replace the historical evidence. The analyses above continue to describe the embedded historical benchmark, so new results are explicitly labeled below.


In [8]:
if RUN_FULL_BENCHMARK:
    if not RUN_LIVE:
        raise ValueError("Set RUN_LIVE = True before enabling the full benchmark.")
    output = ROOT / "runtime/notebook/scheduler_benchmark.json"
    subprocess.run([sys.executable, str(ROOT / "scripts/benchmark_scheduler.py"),
                    "--output", str(output)], cwd=ROOT, check=True)
    fresh = json.loads(output.read_text())
    assert len(fresh["results"]) == 27
    print("FRESH complete benchmark saved to:", output)
    print("New mean QAOA/exact:", mean(r["qaoa"] / r["exact"] for r in fresh["results"]
                                         if r["qaoa"] is not None))
else:
    print("Full benchmark is OFF. Historical benchmark analysis above remains available.")


Full benchmark is OFF. Historical benchmark analysis above remains available.


## 7. What the experiment establishes

The current QAOA simulation can return schedules satisfying the model's rules, but greedy search achieved a higher mean planning score and exact search was faster in the recorded configurations. Changing the initial state improved the fraction of valid quantum samples, which identifies a design choice worth investigating without establishing quantum advantage.

Our contribution is an application formulation and a repeatable comparison framework that helps determine which experiments should come next. We still need an initialization-only control, broader problem instances and stronger classical baselines. Testing scalability would also require avoiding exhaustive cost preprocessing. We have not run quantum hardware experiments or measured improvements in student learning.

## Sources and reproducibility

- Code snapshot: https://github.com/ziyuezhao730/Quantum-Lab/tree/ae69d31018730050a25592d46b6d0fbebd18f03f
- Recorded revised benchmark: `evidence/risi/scheduler_benchmark.json`, source commit `3dfb985e362cee30eaf8b78b8718d9c70ec1a9d3`.
- Earlier benchmark: `evidence/risi/scheduler_initial_benchmark.json`, source commit `598bfa309cc373325276c9e3e5b749b7c5eed0e6`.
- Saved timetable: `evidence/risi/scheduler_example.json`.
- Farhi, Goldstone and Gutmann, *A Quantum Approximate Optimization Algorithm*: https://arxiv.org/abs/1411.4028
- Egger, Marecek and Woerner, *Warm-starting quantum optimization*: https://arxiv.org/abs/2009.10095
- Qiskit Statevector documentation: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector

Our simpler budget-informed initial state addresses a related question to warm-start research, but it does not implement the relaxation-based method in that paper or inherit its guarantees. Embedded data remain previously recorded evidence even after this notebook recomputes their summaries.
